# Suite INS — The transforms, as data

`Types` defines its transforms as mbse-transforms has them: data in a transforms' store
([the port proposal](https://github.com/pitaman71/mbse-codegen-python/blob/main/docs/proposals/port/README.md#each-transform-as-mbse-transforms-has-it)).
`Types.install` writes each transform's prototype, a pattern named `Codegen.<transform>` that declares the transform's
parameters through `Parametrics`; `Transforms.define` writes each transform into the catalog under its name, naming the
procedure registered under it. `Types.INSTALLS` runs, each in a transaction of its own, everything such a store needs.
These cases read the transforms back, and measure what each prototype's spec states: run as a query over a store of
schemas, it keeps the schemas its structural conditions admit, and leaves what needs an expression evaluated to the
caller's verification.

In [ ]:
from mbse.Codegen.Python import Types
from mbse.Programs.Python3 import Parse
from mbse.Schemas.Framework import Builders as B, Parametrics, Plain

TRANSFORMS = ["Dataclass", "Entry", "Union", "Intersection", "Alias", "ListAlias", "DropInlineObject",
              "DropInlineRelation", "DropInlineUnion", "DropInlineIntersection", "DropInlineContainer", "DropUndeclared",
              "Schema"]
transforms = B.Store()  # the transforms' store
for each in Types.INSTALLS:
    transforms.transaction(each)


def parameters(store, prototype):
    """The parameters a prototype declares, by name, in order, each with the names of the schemas its values are of."""
    return {str(e.name): [str(end.schema.name) for end in B.OfParameter.Builder(e.parameter, False, store).for_value]
            for held in B.OfPattern.Builder(prototype, False, store).parameters for e in held.params.parameters}


def book(txn):
    """Schemas of each kind, named `book.*`, each passing or failing a condition of a prototype."""
    S, P = txn.Schemas, txn.Parametrics
    return txn.uid("book/1").mutations([lambda mu: mu.upsert(lambda u: u.exists([lambda p: p
        .symbols({"Contact": S.OfObject, "Phone": S.OfObject, "Holder": S.OfObject, "Holding": S.OfObject,
                  "Owner": S.OfObject, "Listed": S.OfRelation, "Owns": S.OfRelation, "Reach": S.OfUnion,
                  "Channel": S.OfUnion, "Nothing": S.OfUnion, "Vague": S.OfUnion, "Typed": S.OfUnion,
                  "Card": S.OfIntersection, "Blank": S.OfIntersection, "Names": S.OfContainer, "Rows": S.OfContainer})
        .spec([
            lambda _: _.Contact.name("book.Contact").ref(True).properties({
                "name": S.OfStr, "home": S.OfObject(lambda o: o.properties({"street": S.OfStr}))}).adjacencies({
                "directories": lambda a: a.relation([_.Listed]).me("contact")}),  # an inline object held
            lambda _: _.Phone.name("book.Phone").properties({"number": S.OfStr}),
            lambda _: _.Holder.name("book.Holder").properties({"held": S.OfInt}).parameters([P.OfParameters(
                lambda ps: ps.parameters({"T": lambda d: d.parameter(lambda q: q.for_value([S.OfInt]))}))]),
            lambda _: _.Holding.name("book.Holding").properties({"holder": _.Holder}),  # holds what is unbuilt
            lambda _: _.Owner.name("book.Owner").ref(True).adjacencies({
                "owned": lambda a: a.relation([_.Owns]).me("owner")}),  # to a relation declaring parameters
            lambda _: _.Listed.name("book.Listed").links([lambda t: t.name("directory"), lambda t: t.name("contact")]),
            lambda _: _.Owns.name("book.Owns").links([lambda t: t.name("owner"), lambda t: t.name("item")]).parameters([
                P.OfParameters(lambda ps: ps.parameters({"T": lambda d: d.parameter(lambda q: q.for_value([S.OfInt]))}))]),
            lambda _: _.Reach.name("book.Reach").branches([lambda b: b.name("phone").schema([_.Phone])]),
            lambda _: _.Channel.name("book.Channel").flat(True).branches([lambda b: b.name("phone").schema([_.Phone])]),
            lambda _: _.Nothing.name("book.Nothing"),  # no branches
            lambda _: _.Vague.name("book.Vague").flat(None).branches([lambda b: b.name("phone").schema([_.Phone])]),
            lambda _: _.Typed.name("book.Typed").branches([lambda b: b.name("phone").schema([_.Phone])]).parameters([
                P.OfParameters(lambda ps: ps.parameters({"T": lambda d: d.parameter(lambda q: q.for_value([S.OfInt]))}))]),
            lambda _: _.Card.name("book.Card").parts([lambda b: b.name("phone").schema([_.Phone])]),
            lambda _: _.Blank.name("book.Blank"),  # no parts
            lambda _: _.Names.name("book.Names").index([S.OfInt]).element([S.OfStr]),
            lambda _: _.Rows.name("book.Rows").index([S.OfInt]).element([S.OfStr]).parameters([P.OfParameters(
                lambda ps: ps.parameters({"T": lambda d: d.parameter(lambda q: q.for_value([S.OfInt]))}))]),
        ])]))])


def kept(store, prototype):
    """The names of the `book.*` schemas that the prototype named `prototype`, run as a query, binds."""
    held = store.registered(prototype)
    frames = store.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([held]))]))
    return [str(schema.name) for frame in frames for schema in frame.values() if str(schema.name).startswith("book.")]


<a id="INS-01"></a>
## INS-01 · Each transform is found in the catalog by name, naming its procedure and its prototype, `Codegen.<transform>`, whose symbol binds its kind of schema

In [ ]:
with transforms.transaction():
    catalog = transforms.singleton("Transforms.TheCatalog").transforms
    assert [str(e.name) for e in catalog] == TRANSFORMS  # the proposal's table, in its order
    found = {}
    for name in TRANSFORMS:
        transform = catalog.at(name).transform  # by name
        (prototype,) = [e.pattern for e in transform.prototype]
        found[name] = (str(transform.procedure), prototype.name, {s: symbol.schema.name for s, symbol in prototype.symbols.items()})
assert found == {
    "Dataclass": ("Dataclass", "Codegen.Dataclass", {"s": "Schemas.OfObject"}),
    "Entry": ("Entry", "Codegen.Entry", {"r": "Schemas.OfRelation"}),
    "Union": ("Union", "Codegen.Union", {"s": "Schemas.OfUnion"}),
    "Intersection": ("Intersection", "Codegen.Intersection", {"s": "Schemas.OfIntersection"}),
    "Alias": ("Alias", "Codegen.Alias", {"s": "Schemas.OfUnion"}),
    "ListAlias": ("ListAlias", "Codegen.ListAlias", {"s": "Schemas.OfContainer"}),
    "DropInlineObject": ("DropInlineObject", "Codegen.DropInlineObject", {"s": "Schemas.OfObject"}),
    "DropInlineRelation": ("DropInlineRelation", "Codegen.DropInlineRelation", {"r": "Schemas.OfRelation"}),
    "DropInlineUnion": ("DropInlineUnion", "Codegen.DropInlineUnion", {"s": "Schemas.OfUnion"}),
    "DropInlineIntersection": ("DropInlineIntersection", "Codegen.DropInlineIntersection", {"s": "Schemas.OfIntersection"}),
    "DropInlineContainer": ("DropInlineContainer", "Codegen.DropInlineContainer", {"s": "Schemas.OfContainer"}),
    "DropUndeclared": ("DropUndeclared", "Codegen.DropUndeclared", {"r": "Schemas.OfRelation"}),
    "Schema": ("Schema", "Codegen.Schema", {"m": "Programs.Python3.Module"}),
}
assert "NativeAlias" not in found and "DropFormat" not in found and "AliasSchema" not in found  # Decided, 6 and 7


<a id="INS-02"></a>
## INS-02 · Each prototype's parameters and their domains: `frozen` and `slots`, each a `Schemas.OfBool`, where the proposal's table gives them, and none elsewhere

In [ ]:
declared = {name: parameters(transforms, transforms.registered(f"Codegen.{name}")) for name in TRANSFORMS}
flags = {"frozen": ["Schemas.OfBool"], "slots": ["Schemas.OfBool"]}
assert declared == {**{name: {} for name in TRANSFORMS}, "Dataclass": flags, "Entry": {"slots": ["Schemas.OfBool"]},
                    "Union": flags, "Intersection": flags}
assert all(Parametrics.validate(transforms, transforms.registered(f"Codegen.{name}")) == [] for name in TRANSFORMS)


<a id="INS-03"></a>
## INS-03 · Installed again, nothing is written

In [ ]:
before = Plain.ToPlain(transforms).Snapshot()
for each in Types.INSTALLS:
    transforms.transaction(each)
transforms.transaction(Types.install)
assert Plain.ToPlain(transforms).Snapshot() == before


<a id="INS-04"></a>
## INS-04 · The measure: each prototype's spec, run as a query, keeps the schemas its structural conditions admit, and leaves what needs an expression to the caller's verification

In [ ]:
schemas = B.Store()  # a store of schemas, holding the prototypes too
for each in Types.INSTALLS:
    schemas.transaction(each)
schemas.transaction(book)
schemas.transaction(Parse.parse("x = 1\n", "book.py"))  # a module
measured = {name: kept(schemas, f"Codegen.{name}") for name in TRANSFORMS[:-1]}
assert measured == {
    # named by the symbol's kind; no parameters; no adjacency to a relation declaring them. Holding nothing unbuilt
    # at any depth needs an expression: Holding, which holds Holder, is kept
    "Dataclass": ["book.Contact", "book.Phone", "book.Holding"],
    "Entry": ["book.Listed"],  # no parameters; no link declared by an object schema declaring them needs an expression
    "Union": ["book.Reach"],  # flat false or never set, with branches, no parameters: Vague, whose flat is cleared, is not kept
    "Intersection": ["book.Card"],  # with parts, no parameters
    "Alias": ["book.Channel"],  # flat, with branches, no parameters
    "ListAlias": ["book.Names"],  # named, no parameters (Rows declares one); its element and index holding nothing unbuilt needs an expression
    # what a dropped schema is needs an expression: each drop keeps every named schema of its kind
    "DropInlineObject": ["book.Contact", "book.Phone", "book.Holder", "book.Holding", "book.Owner"],
    "DropInlineRelation": ["book.Listed", "book.Owns"],
    "DropInlineUnion": ["book.Reach", "book.Channel", "book.Nothing", "book.Vague", "book.Typed"],
    "DropInlineIntersection": ["book.Card", "book.Blank"],
    "DropInlineContainer": ["book.Names", "book.Rows"],
    "DropUndeclared": ["book.Listed", "book.Owns"],
}
held = schemas.registered("Codegen.Schema")
(module,) = schemas.transaction(lambda t: t.mutations([lambda mu: mu.map(lambda m: m.symbols([held]))]))
assert Plain.ToPlain(schemas)(module["m"].as_schema(), module["m"])["path"]["next"]["entry"]["properties"] == {
    "path": "book.py"}  # every module, by its path in the workspace
